In [1]:
# COMPLETE ML LAB PROGRAM — QUESTIONS (a) TO (f)

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# (a) EDA: EXPLORATORY DATA ANALYSIS
# ============================================================

# Load dataset
df = pd.read_csv("student-mat.csv", sep=";")

print("Dataset shape:", df.shape)
print("\nFirst five rows:\n", df.head())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nBasic statistics:\n", df.describe())

# Analyze G1, G2 and G3
print("\nGrade statistics:\n", df[["G1", "G2", "G3"]].describe())
print("\nCorrelation with G3:\n",
      df[["G1", "G2", "G3"]].corr()["G3"])

# Histograms
df[["G1", "G2", "G3"]].hist(figsize=(10, 4), bins=10)
plt.suptitle("Distribution of G1, G2 and G3")
plt.tight_layout()
plt.show()

# Boxplots
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, col in zip(axes, ["G1", "G2", "G3"]):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f"Boxplot of {col}")

plt.tight_layout()
plt.show()

# Relationship between grades
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.scatterplot(data=df, x="G1", y="G3", ax=axes[0])
axes[0].set_title("G1 vs G3")

sns.scatterplot(data=df, x="G2", y="G3", ax=axes[1])
axes[1].set_title("G2 vs G3")

plt.tight_layout()
plt.show()

# Identify potential outliers using IQR
for col in ["G1", "G2", "G3"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lb = q1 - 1.5 * iqr
    ub = q3 + 1.5 * iqr

    count = ((df[col] < lb) | (df[col] > ub)).sum()
    print(f"{col}: {count} potential outliers")


# ============================================================
# (b) CORRELATION-BASED FEATURE SELECTION
# ============================================================

# Separate input features and target
X = df.drop("G3", axis=1)
y = df["G3"]

# Display correlation heatmap
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

# Split before fitting data-dependent feature selection
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Find highly correlated numerical input features
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
corr_X = X_train[num_cols].corr().abs()

upper = corr_X.where(
    np.triu(np.ones(corr_X.shape), k=1).astype(bool)
)

to_drop = [
    col for col in upper.columns
    if any(upper[col] > 0.90)
]

print("\nRedundant features removed:", to_drop)

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

# Identify numerical and categorical columns
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

# Numerical preprocessing
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])


# ============================================================
# (c) BUILD LINEAR REGRESSION AND DECISION TREE MODELS
# ============================================================

# Linear Regression
lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_pipe.fit(X_train, y_train)

# Decision Tree Regression
dt_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

dt_pipe.fit(X_train, y_train)


# ============================================================
# (d) EVALUATION: MAE, MSE, RMSE, R2
# ============================================================

models = {
    "Linear Regression": lr_pipe,
    "Decision Tree": dt_pipe
}

results = []

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred)
    })

print("\nMODEL EVALUATION:")
print(pd.DataFrame(results).round(4))


# ============================================================
# (e) HYPERPARAMETER TUNING AND 5-FOLD CROSS-VALIDATION
# ============================================================

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Tune Decision Tree using 5-fold CV
params = {
    "model__max_depth": [3, 5, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    dt_pipe,
    params,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Decision Tree parameters:", grid.best_params_)
print("Best CV MAE:", -grid.best_score_)

# Compare 5-fold CV results for both models
scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

cv_models = {
    "Linear Regression": lr_pipe,
    "Decision Tree": grid.best_estimator_
}

for name, model in cv_models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=-1
    )

    print("\n", name)
    print("Mean 5-fold CV MAE:", -scores["test_MAE"].mean())
    print("Mean 5-fold CV MSE:", -scores["test_MSE"].mean())
    print("Mean 5-fold CV RMSE:",
          np.sqrt(-scores["test_MSE"]).mean())
    print("Mean 5-fold CV R2:", scores["test_R2"].mean())


# ============================================================
# (f) BAGGING USING DECISION TREE REGRESSION
# ============================================================

# Best standalone Decision Tree
best_dt = grid.best_estimator_

dt_pred = best_dt.predict(X_test)

# Bagging with Decision Trees
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=100,
    random_state=42
)

bagging.fit(X_train, y_train)
bag_pred = bagging.predict(X_test)

# Compare standalone Decision Tree and Bagging
comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Bagging"],
    "MAE": [
        mean_absolute_error(y_test, dt_pred),
        mean_absolute_error(y_test, bag_pred)
    ],
    "MSE": [
        mean_squared_error(y_test, dt_pred),
        mean_squared_error(y_test, bag_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, dt_pred)),
        np.sqrt(mean_squared_error(y_test, bag_pred))
    ],
    "R2": [
        r2_score(y_test, dt_pred),
        r2_score(y_test, bag_pred)
    ]
})

print("\nDECISION TREE VS BAGGING:")
print(comparison.round(4))

# Final output
print("\nML PIPELINE COMPLETED SUCCESSFULLY")




FileNotFoundError: [Errno 2] No such file or directory: 'student-mat.csv'